# Week 19 -- DPO and the Preference Zoo -- Workbook

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Use a preference margin without losing absolute likelihood. Start with **Exercises 2, 3, 4 and 8** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-use-a-preference-margin-without-losing-absolute-likelihood).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


**Worked examples and derivations:** [Teaching guide](teaching_guide.md) — prerequisite links, analogy boundaries, calculated examples, code connections, and retrieval questions.

In [ ]:
import numpy as np
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import torch, torch.nn.functional as F
torch.set_num_threads(1)
from project import solutions as S
from project.losses import dpo_loss, ipo_loss, simpo_loss, orpo_loss, _log1mexp, implicit_reward_margin
from project.data import ToyPreferenceDataset
# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## Exercise 1 (star) -- the implicit reward
Given `logp = log pi(y|x)` and `logp_ref = log pi_ref(y|x)`, return the implicit reward $\hat r = \beta(\log\pi - \log\pi_\text{ref})$.

> **Hint**: derive it from the closed-form RLHF optimum of week 18; the partition function disappears in any pairwise difference.

In [ ]:
def implicit_reward(logp, logp_ref, beta):
    raise NotImplementedError

In [ ]:
lp = torch.tensor([0.0, -2.0]); lref = torch.tensor([-1.0, -1.0])
r = select_exercise(S.exercise_1_implicit_reward, implicit_reward)(lp, lref, beta=0.5)
assert torch.allclose(r, torch.tensor([0.5, -0.5]))
print('implicit reward =', r)

## Exercise 2 (star star) -- DPO loss from scratch

> **Hint**: `-log sigmoid(beta * ((logp_w - logp_ref_w) - (logp_l - logp_ref_l)))`.  Use `F.logsigmoid`.

In [ ]:
def my_dpo(logp_w, logp_l, logp_ref_w, logp_ref_l, beta=0.1):
    raise NotImplementedError

In [ ]:
z = torch.zeros(4)
loss = select_exercise(S.exercise_2_dpo, my_dpo)(z, z, z, z, beta=0.1)
assert abs(float(loss) - 0.693) < 1e-2
print('DPO loss at equal logits =', float(loss))
fixture=select_exercise(S.exercise_2_dpo, my_dpo)(
    torch.tensor([-1.]),torch.tensor([-2.]),torch.tensor([-2.]),torch.tensor([-2.]),beta=.5)
assert abs(float(fixture)-np.logaddexp(0,-.5))<1e-6


## Exercise 3 (star star star) -- IPO loss from scratch
Squared loss with target margin $1/(2\beta)$.  Explain in a comment why this avoids DPO's saturation pathology.

> **Hint**: `((lr_w - lr_l - 1/(2 beta)) ** 2).mean()`.

In [ ]:
def my_ipo(logp_w, logp_l, logp_ref_w, logp_ref_l, beta=0.1):
    raise NotImplementedError

In [ ]:
z = torch.zeros(4)
loss = select_exercise(S.exercise_3_ipo, my_ipo)(z, z, z, z, beta=0.5)
assert abs(float(loss) - 1.0) < 1e-5
print('IPO loss at margin 0, target 1.0 =', float(loss))
fixture=select_exercise(S.exercise_3_ipo, my_ipo)(
    torch.tensor([-1.]),torch.tensor([-2.]),torch.tensor([-2.]),torch.tensor([-2.]),beta=.5)
assert abs(float(fixture))<1e-6  # margin1 exactly matches the stated IPO target.


## Exercise 4 (star star) -- SimPO loss

> **Hint**: average the log probability per token: `logp / max(1, len)`.

In [ ]:
def my_simpo(logp_w, logp_l, len_w, len_l, beta=2.0, gamma=1.0):
    raise NotImplementedError

In [ ]:
lp_w = torch.tensor([-1.0]); lp_l = torch.tensor([-5.0]); L_w = torch.tensor([1]); L_l = torch.tensor([5])
out = select_exercise(S.exercise_4_simpo, my_simpo)(lp_w, lp_l, L_w, L_l, beta=2.0, gamma=0.0)
assert abs(float(out) - 0.693) < 1e-2
print('SimPO loss at equal avg-logprob =', float(out))
fixture=select_exercise(S.exercise_4_simpo, my_simpo)(
    torch.tensor([-1.]),torch.tensor([-3.]),torch.tensor([1]),torch.tensor([1]),beta=2.,gamma=0.)
assert abs(float(fixture)-np.logaddexp(0,-4.))<1e-6


## Exercise 5 (star star) -- numerically stable log(1 - exp(x))

> **Hint**: use `expm1` for x > log(1/2), otherwise `log1p(-exp(x))`.

In [ ]:
def my_log1mexp(x):
    raise NotImplementedError

In [ ]:
x = torch.tensor([-0.1, -1.0, -5.0])
out = select_exercise(S.exercise_5_log1mexp, my_log1mexp)(x)
naive = torch.log1p(-torch.exp(x))
assert torch.allclose(out, naive, atol=1e-5)
print('log1mexp =', out)

## Exercise 6 (star star) -- ORPO loss

Pass mean completion log probabilities per token to the odds calculation.

> **Hint**: NLL on winners + (-beta * log sigmoid) of the log-odds-ratio gap, using mean completion log probabilities per token.

In [ ]:
def my_orpo(logp_w, logp_l, nll_w, beta=0.1):
    raise NotImplementedError

In [ ]:
lp_w = torch.tensor([-1.0]); lp_l = torch.tensor([-3.0]); nll_w = torch.tensor([1.0])
out = select_exercise(S.exercise_6_orpo, my_orpo)(lp_w, lp_l, nll_w, beta=0.1)
assert torch.isfinite(out)
print('ORPO loss =', float(out))
tie=select_exercise(S.exercise_6_orpo, my_orpo)(
    torch.tensor([-1.]),torch.tensor([-1.]),torch.tensor([1.]),beta=.1)
assert abs(float(tie)-(1+.1*np.log(2)))<1e-6


## Exercise 7 (star star) -- win-rate over two reward arrays

> **Hint**: fraction of items where `rewards_a > rewards_b`.

In [ ]:
def my_win_rate(a, b):
    raise NotImplementedError

In [ ]:
a = torch.tensor([1.0, 2.0, 0.0]); b = torch.tensor([0.0, 1.0, 1.0])
rho = select_exercise(S.exercise_7_win_rate, my_win_rate)(a, b)
assert abs(rho - 2/3) < 1e-6
print('win rate =', rho)

## Exercise 8 (star star) -- implicit-reward margin diagnostic

> **Hint**: a one-line wrapper around the formula in week 18 exercise 7.

In [ ]:
def my_margin(logp_w, logp_l, lref_w, lref_l, beta=0.1):
    raise NotImplementedError

In [ ]:
m = select_exercise(S.exercise_8_margin, my_margin)(
    torch.tensor([-1.0]), torch.tensor([-3.0]),
    torch.tensor([-2.0]), torch.tensor([-2.0]), beta=0.1
)
assert float(m) > 0
print('margin =', float(m))

## Exercise 9 (star star star) -- *Open-ended:* mini bake-off and length-bias diagnostic
Run a *very small* bake-off and inspect the average token length of chosen vs. rejected sequences from the toy dataset.  Larger gap => more length inflation potential.

> **Hint**: feed `lens_chosen` and `lens_rejected` (the response lengths) into `S.exercise_9_length_bias`.

In [ ]:
from project.data import encode
ds = ToyPreferenceDataset(n=32, seed=0)
lens_chosen   = torch.tensor([len(c) for (_, c, _) in ds.pairs])
lens_rejected = torch.tensor([len(l) for (_, _, l) in ds.pairs])
gap = S.exercise_9_length_bias(lens_chosen, lens_rejected)
print('length gap (chosen - rejected) =', gap)
# Negative gap means the *rejected* set is longer -- exactly the dataset we want SimPO on.

## Exercise 10 (star) -- *Meaningful objective:* Plackett--Luce DPO
Derive a DPO loss for a *ranking* over $k$ items, not just pairs.

Mathematical derivation (no code needed; check yourself against the answer below):

Given a ranked tuple $y_{(1)} \succ y_{(2)} \succ \dots \succ y_{(k)}$ for prompt $x$, the Plackett--Luce probability of that ranking under reward $r$ is

$$ P(y_{(1)} \succ \dots \succ y_{(k)}) = \prod_{i=1}^{k} \frac{\exp(r(x, y_{(i)}))}{\sum_{j=i}^{k} \exp(r(x, y_{(j)}))}. $$

Substituting $r = \beta(\log\pi_\theta - \log\pi_\text{ref})$ and taking the negative log gives the loss

$$ \mathcal L_{\text{PL-DPO}} = -\sum_i \left[ \beta(\log\pi_\theta(y_{(i)}|x) - \log\pi_\text{ref}(y_{(i)}|x)) - \log \sum_{j \ge i} e^{\beta(\log\pi_\theta(y_{(j)}|x) - \log\pi_\text{ref}(y_{(j)}|x))} \right]. $$

When $k=2$ this collapses to the pairwise DPO loss; for $k>2$ it generalises naturally and is the loss form used by *ranked-DPO* and the multi-candidate ATS suggestion-ranking setting.

## Wrap-up
You have implemented (or read) every DPO-family loss in the modern literature.
On a real GPU, `project/trl_recipe.py` (sketched in `notes.tex`) extends this to Qwen-2.5-0.5B + LoRA on a slice of UltraFeedback.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (star) -- the implicit reward</summary>

> **Hint**: derive it from the closed-form RLHF optimum of week 18; the partition function disappears in any pairwise difference.

</details>

<details>
<summary>Exercise 2 (star star) -- DPO loss from scratch</summary>

> **Hint**: `-log sigmoid(beta * ((logp_w - logp_ref_w) - (logp_l - logp_ref_l)))`.  Use `F.logsigmoid`.

</details>

<details>
<summary>Exercise 3 (star star star) -- IPO loss from scratch</summary>

> **Hint**: `((lr_w - lr_l - 1/(2 beta)) ** 2).mean()`.

</details>

<details>
<summary>Exercise 4 (star star) -- SimPO loss</summary>

> **Hint**: average the log probability per token: `logp / max(1, len)`.

</details>

<details>
<summary>Exercise 5 (star star) -- numerically stable log(1 - exp(x))</summary>

> **Hint**: use `expm1` for x > log(1/2), otherwise `log1p(-exp(x))`.

</details>

<details>
<summary>Exercise 6 (star star) -- ORPO loss</summary>

> **Hint**: NLL on winners + (-beta * log sigmoid) of the log-odds-ratio gap, using mean completion log probabilities per token.

</details>

<details>
<summary>Exercise 7 (star star) -- win-rate over two reward arrays</summary>

> **Hint**: fraction of items where `rewards_a > rewards_b`.

</details>

<details>
<summary>Exercise 8 (star star) -- implicit-reward margin diagnostic</summary>

> **Hint**: a one-line wrapper around the formula in week 18 exercise 7.

</details>

<details>
<summary>Exercise 9 (star star star) -- *Open-ended:* mini bake-off and length-bias diagnostic</summary>

> **Hint**: feed `lens_chosen` and `lens_rejected` (the response lengths) into `S.exercise_9_length_bias`.

</details>

<details>
<summary>Exercise 10 (star) -- *Meaningful objective:* Plackett--Luce DPO</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Statistical investigation — Preference losses identify a relative comparison

**Goal.** Separate pairwise loss geometry from calibrated preference and utility claims. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** For a pairwise preference model, P(chosen preferred)=sigmoid(r_chosen-r_rejected).
Predict what happens if the same constant is added to both rewards. For a DPO
log-ratio margin m, compare softplus(-βm) at several β values. A lower training
loss is not itself an absolute utility or held-out win-rate estimate.

**Work a small case by hand.** A reward pair (2,0.5) has difference 1.5 and modeled preference
sigmoid(1.5)≈0.8176. Adding any shared constant leaves that difference
unchanged. For the DPO pair objective L=log(1+exp(-beta m)), m=0 gives
L=log(2)≈0.6931 for every beta. At m=1,beta=1, L≈0.3133; increasing
beta to 2 gives L≈0.1269. Different loss scales do not constitute
different observed win rates.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/06-regression-validation.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
margins=np.linspace(-4,4,150);betas=[.25,1.,2.]
fig,axes=plt.subplots(1,2,figsize=(9,3.3))
for beta in betas:
    loss=np.logaddexp(0,-beta*margins)
    axes[0].plot(margins,loss,label=f'beta={beta}')
    axes[1].plot(margins,1/(1+np.exp(-beta*margins)),label=f'beta={beta}')
r_chosen,r_rejected=2.,.5
for offset in [0,10,-20]:
    preference=1/(1+np.exp(-((r_chosen+offset)-(r_rejected+offset))))
    print('Reward offset:',offset,'pair preference probability:',preference)
assert np.isclose(np.logaddexp(0,0),np.log(2))
axes[0].set(xlabel='Policy-versus-reference pair log-ratio margin m',ylabel='DPO pair loss')
axes[1].set(xlabel='Raw pair log-ratio margin m',ylabel='Modeled preference probability\nsigmoid(beta m)')
for ax in axes:ax.legend(fontsize=8)
fig.tight_layout();plt.show()


**Interpret and transfer.** Pair differences are invariant to a shared additive reward shift. The plots
describe the stated model and objective; model probabilities require calibration
against independent labels before being interpreted as empirical preference rates.

**Transfer.** Evaluate on unseen prompts with several annotators per prompt.
State how ties, annotator disagreement and prompt clusters enter the metric;
keep β selection separate from the final comparison.


**Read your result.** The right panel plots sigmoid(beta m) against the raw margin m. Its curves
change because beta rescales the sigmoid argument; they describe the stated
model rather than calibrated probabilities measured from held-out people.

<details><summary>Worked feedback for the changed assumption</summary>

Freeze beta using validation prompts, then evaluate unseen prompts with a
prespecified win/tie rule and paired policy outputs. Prompt-level resampling
preserves repeated annotations. If the claim generalizes across annotators
as well, shared annotator effects require an additional crossed-design
analysis; treating every label as an independent new prompt is unjustified.

</details>
